In [0]:
# ==========================================================
# TSI - FASE 6: EXPERIMENTACIÓN CON MLFLOW
# PASO 58.3B: CORRECCIÓN DE FRONTERAS TEMPORALES
# ==========================================================

import pandas as pd

print("=== CONTEOS ANTES DE LA CORRECCIÓN ===")

print(f"Train: {len(df_ridge_train):,}")
print(f"Validación: {len(df_ridge_val):,}")
print(f"Test: {len(df_test_pd):,}")

# 1. Identificar las fronteras temporales
inicio_validacion = df_ridge_val["hora"].min()
inicio_test = df_test_pd["hora"].min()

print("\n=== FRONTERAS ===")
print(f"Inicio validación: {inicio_validacion}")
print(f"Inicio test: {inicio_test}")

# 2. Excluir pares de entrenamiento cuyo objetivo
# alcance el periodo de validación
df_ridge_train = (
    df_ridge_train[
        df_ridge_train["hora_siguiente"] < inicio_validacion
    ]
    .copy()
)

# 3. Excluir pares de validación cuyo objetivo
# alcance el periodo de prueba
df_ridge_val = (
    df_ridge_val[
        df_ridge_val["hora_siguiente"] < inicio_test
    ]
    .copy()
)

# 4. Verificar separación temporal estricta
assert (
    df_ridge_train["hora_siguiente"].max()
    < df_ridge_val["hora"].min()
)

assert (
    df_ridge_val["hora_siguiente"].max()
    < df_test_pd["hora"].min()
)

# 5. Verificar integridad
FEATURES = [
    "tsi_promedio",
    "tsi_desviacion",
    "observaciones",
    "segmentos",
    "hora_dia",
    "dia_semana"
]

TARGET = "tsi_siguiente"

for nombre, df in [
    ("Train", df_ridge_train),
    ("Validación", df_ridge_val),
    ("Test", df_test_pd)
]:
    assert len(df) > 0
    assert df[FEATURES + [TARGET]].isna().sum().sum() == 0

print("\n=== CONTEOS DESPUÉS DE LA CORRECCIÓN ===")

print(f"Train: {len(df_ridge_train):,}")
print(f"Validación: {len(df_ridge_val):,}")
print(f"Test: {len(df_test_pd):,}")

print("\n=== VALIDACIÓN FINAL ===")
print("Sin objetivos compartidos entre particiones.")
print("Sin valores nulos.")
print("Separación temporal estricta validada.")
print("No se modificaron tablas Delta.")

In [0]:
# ==========================================================
# TSI - FASE 6: EXPERIMENTACIÓN CON MLFLOW
# PASO 58.4: ENTRENAMIENTO Y VALIDACIÓN DE RIDGE
# ==========================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=== PREPARACIÓN DE VARIABLES ===")

FEATURES = [
    "tsi_promedio",
    "tsi_desviacion",
    "observaciones",
    "segmentos",
    "hora_dia",
    "dia_semana"
]

TARGET = "tsi_siguiente"

X_train = df_ridge_train[FEATURES]
y_train = df_ridge_train[TARGET]

X_val = df_ridge_val[FEATURES]
y_val = df_ridge_val[TARGET]

assert len(X_train) == 2775
assert len(X_val) == 695

print(f"Entrenamiento: {len(X_train):,}")
print(f"Validación: {len(X_val):,}")

# 1. Valores de regularización
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]

resultados = []
modelos = {}

print("\n=== ENTRENAMIENTO RIDGE ===")

for alpha in alphas:

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha))
    ])

    # Ajustar solamente con entrenamiento
    pipeline.fit(X_train, y_train)

    # Evaluar exclusivamente en validación
    predicciones = pipeline.predict(X_val)

    mae = mean_absolute_error(y_val, predicciones)
    rmse = np.sqrt(
        mean_squared_error(y_val, predicciones)
    )
    r2 = r2_score(y_val, predicciones)

    resultados.append({
        "alpha": alpha,
        "mae_val": mae,
        "rmse_val": rmse,
        "r2_val": r2
    })

    modelos[alpha] = pipeline

# 2. Comparación de hiperparámetros
df_resultados_ridge = (
    pd.DataFrame(resultados)
    .sort_values("mae_val")
    .reset_index(drop=True)
)

print("\n=== RESULTADOS DE VALIDACIÓN ===")
print(df_resultados_ridge.to_string(index=False))

# 3. Seleccionar mejor modelo según MAE
mejor_alpha = float(
    df_resultados_ridge.iloc[0]["alpha"]
)

mejor_ridge = modelos[mejor_alpha]

print("\n=== MODELO SELECCIONADO ===")
print(f"Mejor alpha: {mejor_alpha}")
print(
    "MAE validación:",
    round(df_resultados_ridge.iloc[0]["mae_val"], 4)
)

# 4. Comparar con persistencia en el mismo periodo
pred_persistencia_val = (
    df_ridge_val["tsi_promedio"].to_numpy()
)

mae_persistencia_val = mean_absolute_error(
    y_val,
    pred_persistencia_val
)

mae_ridge_val = float(
    df_resultados_ridge.iloc[0]["mae_val"]
)

print("\n=== COMPARACIÓN EN VALIDACIÓN ===")
print(f"Persistencia MAE: {mae_persistencia_val:.4f}")
print(f"Ridge MAE: {mae_ridge_val:.4f}")

if mae_ridge_val < mae_persistencia_val:
    print("Ridge supera a persistencia en validación.")
else:
    print("Ridge no supera a persistencia en validación.")

print("\nEntrenamiento y selección completados.")
print("El conjunto de prueba todavía no se utilizó.")
print("Todavía no se registraron nuevos runs en MLflow.")

In [0]:
# ==========================================================
# TSI - FASE 6: EXPERIMENTACIÓN CON MLFLOW
# PASO 58.5: EVALUACIÓN FINAL DE RIDGE Y REGISTRO
# ==========================================================

import mlflow
import mlflow.sklearn
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=== EVALUACIÓN RIDGE EN TEST ===")

# 1. Preparar datos de prueba
X_test = df_test_pd[FEATURES]
y_test = df_test_pd[TARGET]

# 2. Evaluar modelo seleccionado en validación
pred_ridge = mejor_ridge.predict(X_test)

mae_ridge = mean_absolute_error(y_test, pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, pred_ridge))
r2_ridge = r2_score(y_test, pred_ridge)

# 3. Baseline sobre las mismas observaciones
pred_baseline = df_test_pd["tsi_promedio"].to_numpy()

mae_baseline = mean_absolute_error(y_test, pred_baseline)
rmse_baseline = np.sqrt(
    mean_squared_error(y_test, pred_baseline)
)
r2_baseline = r2_score(y_test, pred_baseline)

print("\n=== COMPARACIÓN EN TEST ===")

print(
    f"Persistencia | MAE: {mae_baseline:.4f} "
    f"| RMSE: {rmse_baseline:.4f} "
    f"| R²: {r2_baseline:.4f}"
)

print(
    f"Ridge        | MAE: {mae_ridge:.4f} "
    f"| RMSE: {rmse_ridge:.4f} "
    f"| R²: {r2_ridge:.4f}"
)

mejora_pct = (
    (mae_baseline - mae_ridge)
    / mae_baseline
) * 100

print(f"\nMejora relativa MAE: {mejora_pct:.2f}%")

assert len(pred_ridge) == 869
assert np.isfinite(
    [mae_ridge, rmse_ridge, r2_ridge]
).all()

# 4. Configurar MLflow
EXPERIMENTO = "/Shared/TSI_PEMS_BAY_Pronostico_Horario"

mlflow.set_experiment(EXPERIMENTO)

print("\n=== REGISTRO EN MLFLOW ===")

with mlflow.start_run(
    run_name="ridge_alpha_10_v1"
) as run:

    mlflow.log_params({
        "proyecto": "Traffic Stability Index",
        "fuente": "pems_bay",
        "modelo": "ridge",
        "alpha": float(mejor_alpha),
        "criterio_seleccion": "mae_validacion",
        "horizonte_horas": 1,
        "normalizacion": "percentiles_train",
        "p05_train": 40.3,
        "p95_train": 69.8,
        "train_registros": len(df_ridge_train),
        "validacion_registros": len(df_ridge_val),
        "test_registros": len(df_test_pd),
        "version_tsi": "TSI_TRAIN_NORM_V1"
    })

    mlflow.log_metrics({
        "mae_test": float(mae_ridge),
        "rmse_test": float(rmse_ridge),
        "r2_test": float(r2_ridge),
        "mae_validacion": float(mae_ridge_val),
        "mae_baseline_test": float(mae_baseline),
        "mejora_mae_pct": float(mejora_pct)
    })

    # Guardar el Pipeline entrenado
    mlflow.sklearn.log_model(
        sk_model=mejor_ridge,
        name="modelo_ridge"
    )

    run_id = run.info.run_id

print(f"Experimento: {EXPERIMENTO}")
print(f"Run ID: {run_id}")
print("Modelo Ridge y métricas registrados.")

print("\nPASO 58.5 COMPLETADO")

In [0]:
# ==========================================================
# TSI - FASE 6: EXPERIMENTACIÓN CON MLFLOW
# PASO 58.6: RANDOM FOREST - ENTRENAMIENTO Y VALIDACIÓN
# ==========================================================

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=== PREPARACIÓN RANDOM FOREST ===")

FEATURES = [
    "tsi_promedio",
    "tsi_desviacion",
    "observaciones",
    "segmentos",
    "hora_dia",
    "dia_semana"
]

TARGET = "tsi_siguiente"

X_train = df_ridge_train[FEATURES]
y_train = df_ridge_train[TARGET]

X_val = df_ridge_val[FEATURES]
y_val = df_ridge_val[TARGET]

assert len(X_train) == 2775
assert len(X_val) == 695

# 1. Configuraciones experimentales
configuraciones = [
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 100, "max_depth": 10},
    {"n_estimators": 200, "max_depth": 5},
    {"n_estimators": 200, "max_depth": 10}
]

resultados_rf = []
modelos_rf = {}

print("\n=== ENTRENAMIENTO RANDOM FOREST ===")

for config in configuraciones:

    modelo = RandomForestRegressor(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    modelo.fit(X_train, y_train)

    predicciones = modelo.predict(X_val)

    mae = mean_absolute_error(y_val, predicciones)
    rmse = np.sqrt(
        mean_squared_error(y_val, predicciones)
    )
    r2 = r2_score(y_val, predicciones)

    clave = (
        config["n_estimators"],
        config["max_depth"]
    )

    modelos_rf[clave] = modelo

    resultados_rf.append({
        "n_estimators": config["n_estimators"],
        "max_depth": config["max_depth"],
        "mae_val": mae,
        "rmse_val": rmse,
        "r2_val": r2
    })

# 2. Ordenar por MAE de validación
df_resultados_rf = (
    pd.DataFrame(resultados_rf)
    .sort_values("mae_val")
    .reset_index(drop=True)
)

print("\n=== RESULTADOS DE VALIDACIÓN ===")
print(df_resultados_rf.to_string(index=False))

# 3. Seleccionar mejor configuración
mejor_config_rf = df_resultados_rf.iloc[0]

mejor_n_estimators = int(
    mejor_config_rf["n_estimators"]
)

mejor_max_depth = int(
    mejor_config_rf["max_depth"]
)

mejor_rf = modelos_rf[
    (mejor_n_estimators, mejor_max_depth)
]

mae_rf_val = float(
    mejor_config_rf["mae_val"]
)

# 4. Comparar con persistencia y Ridge
mae_persistencia_val = mean_absolute_error(
    y_val,
    df_ridge_val["tsi_promedio"]
)

print("\n=== COMPARACIÓN EN VALIDACIÓN ===")
print(f"Persistencia MAE: {mae_persistencia_val:.4f}")
print(f"Ridge MAE:        {mae_ridge_val:.4f}")
print(f"Random Forest MAE:{mae_rf_val:.4f}")

print("\n=== MEJOR CONFIGURACIÓN RANDOM FOREST ===")
print(f"n_estimators: {mejor_n_estimators}")
print(f"max_depth: {mejor_max_depth}")
print("min_samples_leaf: 5")

assert np.isfinite(
    df_resultados_rf[
        ["mae_val", "rmse_val", "r2_val"]
    ].to_numpy()
).all()

print("\nEntrenamiento y selección completados.")
print("El conjunto de prueba todavía no se utilizó.")
print("No se registraron nuevos runs en MLflow.")

In [0]:
# ==========================================================
# TSI - FASE 6: EXPERIMENTACIÓN CON MLFLOW
# PASO 58.7: EVALUACIÓN RANDOM FOREST Y REGISTRO MLFLOW
# ==========================================================

import mlflow
import mlflow.sklearn
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=== EVALUACIÓN RANDOM FOREST EN TEST ===")

# 1. Preparar conjunto de prueba
X_test = df_test_pd[FEATURES]
y_test = df_test_pd[TARGET]

# 2. Predicciones del modelo seleccionado
pred_rf = mejor_rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))
r2_rf = r2_score(y_test, pred_rf)

# 3. Baseline sobre las mismas observaciones
pred_baseline = df_test_pd["tsi_promedio"].to_numpy()

mae_baseline = mean_absolute_error(y_test, pred_baseline)
rmse_baseline = np.sqrt(
    mean_squared_error(y_test, pred_baseline)
)
r2_baseline = r2_score(y_test, pred_baseline)

# 4. Comparación con Ridge ya evaluado
print("\n=== COMPARACIÓN DE MODELOS EN TEST ===")

print(
    f"Persistencia | MAE: {mae_baseline:.4f} "
    f"| RMSE: {rmse_baseline:.4f} "
    f"| R²: {r2_baseline:.4f}"
)

print(
    f"Ridge        | MAE: {mae_ridge:.4f} "
    f"| RMSE: {rmse_ridge:.4f} "
    f"| R²: {r2_ridge:.4f}"
)

print(
    f"Random Forest| MAE: {mae_rf:.4f} "
    f"| RMSE: {rmse_rf:.4f} "
    f"| R²: {r2_rf:.4f}"
)

mejora_rf_pct = (
    (mae_baseline - mae_rf)
    / mae_baseline
) * 100

print(
    f"\nMejora MAE frente a persistencia: "
    f"{mejora_rf_pct:.2f}%"
)

assert len(pred_rf) == 869
assert np.isfinite(
    [mae_rf, rmse_rf, r2_rf]
).all()

# 5. Configurar experimento MLflow
EXPERIMENTO = "/Shared/TSI_PEMS_BAY_Pronostico_Horario"

mlflow.set_experiment(EXPERIMENTO)

print("\n=== REGISTRO EN MLFLOW ===")

with mlflow.start_run(
    run_name="random_forest_v1"
) as run:

    mlflow.log_params({
        "proyecto": "Traffic Stability Index",
        "fuente": "pems_bay",
        "modelo": "random_forest",
        "n_estimators": int(mejor_n_estimators),
        "max_depth": int(mejor_max_depth),
        "min_samples_leaf": 5,
        "random_state": 42,
        "horizonte_horas": 1,
        "criterio_seleccion": "mae_validacion",
        "normalizacion": "percentiles_train",
        "p05_train": 40.3,
        "p95_train": 69.8,
        "train_registros": len(df_ridge_train),
        "validacion_registros": len(df_ridge_val),
        "test_registros": len(df_test_pd),
        "version_tsi": "TSI_TRAIN_NORM_V1"
    })

    mlflow.log_metrics({
        "mae_test": float(mae_rf),
        "rmse_test": float(rmse_rf),
        "r2_test": float(r2_rf),
        "mae_validacion": float(mae_rf_val),
        "mae_baseline_test": float(mae_baseline),
        "mejora_mae_pct": float(mejora_rf_pct)
    })

    # Registrar modelo con ejemplo de entrada
    # para inferir su firma automáticamente.
    mlflow.sklearn.log_model(
        sk_model=mejor_rf,
        name="modelo_random_forest",
        input_example=X_train.head(5)
    )

    run_id_rf = run.info.run_id

print(f"Experimento: {EXPERIMENTO}")
print(f"Run ID: {run_id_rf}")
print("Modelo Random Forest y métricas registrados.")

print("\nPASO 58.7 COMPLETADO")

In [0]:
# ==========================================================
# TSI - FASE 6: EXPERIMENTACIÓN CON MLFLOW
# PASO 58.8: DIAGNÓSTICO RANDOM FOREST
# ==========================================================

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error

print("=== IMPORTANCIA DE VARIABLES ===")

df_importancia_rf = (
    pd.DataFrame({
        "variable": FEATURES,
        "importancia": mejor_rf.feature_importances_
    })
    .sort_values("importancia", ascending=False)
    .reset_index(drop=True)
)

print(df_importancia_rf.to_string(index=False))

assert np.isclose(
    df_importancia_rf["importancia"].sum(),
    1.0
)

# 1. Construir diagnóstico de predicciones
df_diagnostico_rf = df_test_pd[
    ["hora", "hora_siguiente", "tsi_promedio", TARGET]
].copy()

df_diagnostico_rf["prediccion_rf"] = pred_rf

df_diagnostico_rf["prediccion_persistencia"] = (
    df_diagnostico_rf["tsi_promedio"]
)

df_diagnostico_rf["error_abs_rf"] = (
    df_diagnostico_rf[TARGET]
    - df_diagnostico_rf["prediccion_rf"]
).abs()

df_diagnostico_rf["error_abs_persistencia"] = (
    df_diagnostico_rf[TARGET]
    - df_diagnostico_rf["prediccion_persistencia"]
).abs()

# 2. Diagnóstico por hora del día
df_diagnostico_rf["hora_dia"] = (
    pd.to_datetime(df_diagnostico_rf["hora"]).dt.hour
)

resumen_horario = (
    df_diagnostico_rf
    .groupby("hora_dia")
    .agg(
        observaciones=("error_abs_rf", "size"),
        mae_rf=("error_abs_rf", "mean"),
        mae_persistencia=("error_abs_persistencia", "mean")
    )
    .reset_index()
)

print("\n=== ERROR POR HORA DEL DÍA ===")
print(resumen_horario.round(4).to_string(index=False))

# 3. Mayores errores del modelo
print("\n=== LOS 10 MAYORES ERRORES DE RANDOM FOREST ===")

columnas = [
    "hora_siguiente",
    "tsi_siguiente",
    "prediccion_rf",
    "error_abs_rf"
]

print(
    df_diagnostico_rf
    .nlargest(10, "error_abs_rf")[columnas]
    .round(4)
    .to_string(index=False)
)

# 4. Verificación global
print("\n=== VERIFICACIÓN FINAL ===")

print(f"Registros evaluados: {len(df_diagnostico_rf):,}")

print(
    "MAE Random Forest:",
    round(
        mean_absolute_error(
            df_diagnostico_rf[TARGET],
            df_diagnostico_rf["prediccion_rf"]
        ),
        4
    )
)

print("Diagnóstico completado.")
print("No se modificó ni reentrenó el modelo.")
print("No se crearon nuevos runs en MLflow.")

In [0]:
# ==========================================================
# TSI - FASE 6: VALIDACIÓN METODOLÓGICA
# PASO 58.9: AUDITORÍA DE PERCENTILES TEMPORALES
# ==========================================================

from pyspark.sql import functions as F

print("=== AUDITORÍA DE NORMALIZACIÓN TEMPORAL ===")

# 1. Tabla original de modelado
TABLA_ORIGEN = "workspace.tsi.international_modeling"

# 2. Identificar fronteras temporales
inicio_validacion = df_ridge_val["hora"].min()
inicio_test = df_test_pd["hora"].min()

print(f"Inicio validación: {inicio_validacion}")
print(f"Inicio prueba: {inicio_test}")

# 3. Seleccionar exclusivamente datos de PEMS-BAY
# anteriores al inicio de validación
df_origen_train_interno = (
    spark.table(TABLA_ORIGEN)
    .filter(F.col("fuente") == "pems_bay")
    .filter(
        F.col("timestamp")
        < F.lit(inicio_validacion.to_pydatetime())
    )
    .filter(F.col("valor_principal").isNotNull())
    .filter(F.col("valor_principal") > 0)
)

# 4. Calcular estadísticas y percentiles
resultado = (
    df_origen_train_interno
    .agg(
        F.count("*").alias("observaciones"),
        F.min("timestamp").alias("fecha_min"),
        F.max("timestamp").alias("fecha_max"),
        F.expr(
            "percentile_approx("
            "valor_principal, "
            "array(0.05, 0.95), "
            "10000)"
        ).alias("percentiles")
    )
    .collect()[0]
)

# 5. Recuperar percentiles
p05_interno = float(resultado["percentiles"][0])
p95_interno = float(resultado["percentiles"][1])

print("\n=== RESULTADOS ===")

print(f"Observaciones: {resultado['observaciones']:,}")
print(f"Primera fecha: {resultado['fecha_min']}")
print(f"Última fecha: {resultado['fecha_max']}")

print(f"\nP05 entrenamiento interno: {p05_interno:.4f}")
print(f"P95 entrenamiento interno: {p95_interno:.4f}")

# 6. Comparar contra los percentiles anteriores
P05_ANTERIOR = 40.3
P95_ANTERIOR = 69.8

print("\n=== COMPARACIÓN CON NORMALIZACIÓN ANTERIOR ===")

print(f"P05 anterior: {P05_ANTERIOR:.4f}")
print(f"P95 anterior: {P95_ANTERIOR:.4f}")

print(
    f"Diferencia P05: "
    f"{p05_interno - P05_ANTERIOR:+.4f}"
)

print(
    f"Diferencia P95: "
    f"{p95_interno - P95_ANTERIOR:+.4f}"
)

# 7. Validaciones
assert resultado["observaciones"] > 0, (
    "No se encontraron observaciones."
)

assert p05_interno < p95_interno, (
    "Los percentiles calculados no son válidos."
)

assert resultado["fecha_max"] < inicio_validacion, (
    "Se detectaron datos fuera del periodo permitido."
)

print("\n=== VALIDACIÓN FINAL ===")
print("Percentiles calculados correctamente.")
print("Sin datos posteriores al inicio de validación.")
print("No se modificaron tablas Delta.")
print("No se entrenaron modelos.")
print("No se registraron nuevos runs en MLflow.")

print("\nPASO 58.9 COMPLETADO")

In [0]:
# ==========================================================
# TSI - FASE 6: CORRECCIÓN METODOLÓGICA
# PASO 59.1: RECONSTRUCCIÓN DEL TSI HORARIO V2
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql import Window

print("=== RECONSTRUCCIÓN TSI V2 ===")

TABLA_ORIGEN = "workspace.tsi.international_modeling"

P05_V2 = p05_interno
P95_V2 = p95_interno

assert P05_V2 == 41.5
assert P95_V2 == 69.8

# 1. Recuperar los datos de PEMS-BAY
df_pems_v2 = (
    spark.table(TABLA_ORIGEN)
    .filter(F.col("fuente") == "pems_bay")
    .filter(F.col("valor_principal").isNotNull())
    .filter(F.col("valor_principal") > 0)
)

# 2. Normalizar utilizando percentiles
# calculados exclusivamente en entrenamiento interno
df_pems_v2 = df_pems_v2.withColumn(
    "tsi_v2",
    F.greatest(
        F.lit(0.0),
        F.least(
            F.lit(100.0),
            100.0 * (
                F.col("valor_principal") - F.lit(P05_V2)
            ) / F.lit(P95_V2 - P05_V2)
        )
    )
)

# 3. Agregar observaciones por hora
df_horario_v2 = (
    df_pems_v2
    .withColumn(
        "hora",
        F.date_trunc("hour", F.col("timestamp"))
    )
    .groupBy("hora")
    .agg(
        F.avg("tsi_v2").alias("tsi_promedio"),
        F.stddev("tsi_v2").alias("tsi_desviacion"),
        F.count("*").alias("observaciones"),
        F.countDistinct("segmento_id").alias("segmentos")
    )
    .orderBy("hora")
)

# 4. Crear pares de pronóstico a una hora
ventana = Window.orderBy("hora")

df_pares_v2 = (
    df_horario_v2
    .withColumn(
        "hora_siguiente",
        F.lead("hora").over(ventana)
    )
    .withColumn(
        "tsi_siguiente",
        F.lead("tsi_promedio").over(ventana)
    )
    .filter(
        F.col("hora_siguiente")
        == F.expr("hora + INTERVAL 1 HOUR")
    )
)

# 5. Dividir por fechas, respetando las fronteras
# temporales estrictas de la versión anterior
INICIO_VALIDACION = "2017-04-26 18:00:00"
INICIO_TEST = "2017-05-25 18:00:00"

df_train_v2 = df_pares_v2.filter(
    F.col("hora_siguiente") < F.lit(INICIO_VALIDACION)
)

df_val_v2 = df_pares_v2.filter(
    (F.col("hora") >= F.lit(INICIO_VALIDACION))
    & (F.col("hora_siguiente") < F.lit(INICIO_TEST))
)

df_test_v2 = df_pares_v2.filter(
    F.col("hora") >= F.lit(INICIO_TEST)
)

# 6. Verificar cantidad de registros
n_train_v2 = df_train_v2.count()
n_val_v2 = df_val_v2.count()
n_test_v2 = df_test_v2.count()

print("\n=== PARTICIONES TSI V2 ===")

print(f"Entrenamiento: {n_train_v2:,}")
print(f"Validación: {n_val_v2:,}")
print(f"Prueba: {n_test_v2:,}")

# 7. Validar integridad de los datos
assert n_train_v2 == 2775
assert n_val_v2 == 695
assert n_test_v2 == 869

for nombre, df in [
    ("train", df_train_v2),
    ("validacion", df_val_v2),
    ("test", df_test_v2)
]:
    nulos = df.filter(
        F.col("tsi_promedio").isNull()
        | F.col("tsi_desviacion").isNull()
        | F.col("tsi_siguiente").isNull()
    ).count()

    assert nulos == 0, (
        f"Valores nulos en {nombre}: {nulos}"
    )

# 8. Comprobar fronteras temporales
assert (
    df_train_v2.agg(
        F.max("hora_siguiente")
    ).first()[0]
    <
    df_val_v2.agg(
        F.min("hora")
    ).first()[0]
)

assert (
    df_val_v2.agg(
        F.max("hora_siguiente")
    ).first()[0]
    <
    df_test_v2.agg(
        F.min("hora")
    ).first()[0]
)

print("\n=== VALIDACIÓN FINAL ===")
print(f"Normalización: P05={P05_V2}, P95={P95_V2}")
print("Separación temporal validada.")
print("Sin valores nulos.")
print("Sin modificación de tablas Delta.")
print("Sin nuevos experimentos MLflow.")

print("\nPASO 59.1 COMPLETADO")

In [0]:
# ==========================================================
# TSI - FASE 6: CORRECCIÓN METODOLÓGICA
# PASO 59.2: PREPARACIÓN DE DATOS PARA MODELADO V2
# ==========================================================

import pandas as pd
import numpy as np

print("=== PREPARACIÓN DE DATOS TSI V2 ===")

# 1. Convertir particiones Spark a Pandas
df_train_v2_pd = (
    df_train_v2.orderBy("hora").toPandas()
)

df_val_v2_pd = (
    df_val_v2.orderBy("hora").toPandas()
)

df_test_v2_pd = (
    df_test_v2.orderBy("hora").toPandas()
)

# 2. Crear características temporales
for df in [
    df_train_v2_pd,
    df_val_v2_pd,
    df_test_v2_pd
]:
    df["hora"] = pd.to_datetime(df["hora"])
    df["hora_siguiente"] = pd.to_datetime(
        df["hora_siguiente"]
    )

    df["hora_dia"] = df["hora"].dt.hour
    df["dia_semana"] = df["hora"].dt.dayofweek

# 3. Definir características y objetivo
FEATURES_V2 = [
    "tsi_promedio",
    "tsi_desviacion",
    "observaciones",
    "segmentos",
    "hora_dia",
    "dia_semana"
]

TARGET_V2 = "tsi_siguiente"

# 4. Construir matrices de entrenamiento
X_train_v2 = df_train_v2_pd[FEATURES_V2]
y_train_v2 = df_train_v2_pd[TARGET_V2]

X_val_v2 = df_val_v2_pd[FEATURES_V2]
y_val_v2 = df_val_v2_pd[TARGET_V2]

X_test_v2 = df_test_v2_pd[FEATURES_V2]
y_test_v2 = df_test_v2_pd[TARGET_V2]

# 5. Validar integridad
print("\n=== VALIDACIÓN DE PARTICIONES ===")

for nombre, X, y in [
    ("Train", X_train_v2, y_train_v2),
    ("Validación", X_val_v2, y_val_v2),
    ("Test", X_test_v2, y_test_v2)
]:
    nulos = (
        X.isna().sum().sum()
        + y.isna().sum()
    )

    print(
        f"{nombre}: {len(X):,} registros "
        f"| nulos: {nulos}"
    )

    assert nulos == 0
    assert np.isfinite(X.to_numpy()).all()
    assert np.isfinite(y.to_numpy()).all()

assert len(X_train_v2) == 2775
assert len(X_val_v2) == 695
assert len(X_test_v2) == 869

# 6. Verificar fronteras temporales
assert (
    df_train_v2_pd["hora_siguiente"].max()
    < df_val_v2_pd["hora"].min()
)

assert (
    df_val_v2_pd["hora_siguiente"].max()
    < df_test_v2_pd["hora"].min()
)

# 7. Comparar TSI V1 y V2 en el periodo de prueba
comparacion = pd.DataFrame({
    "tsi_v1": df_test_pd["tsi_promedio"].to_numpy(),
    "tsi_v2": df_test_v2_pd["tsi_promedio"].to_numpy()
})

diferencia_media = (
    comparacion["tsi_v2"] - comparacion["tsi_v1"]
).abs().mean()

print("\n=== COMPARACIÓN DE NORMALIZACIÓN ===")
print(f"Diferencia absoluta media V1 vs V2: {diferencia_media:.4f}")

print("\n=== VARIABLES DE MODELADO ===")
print(FEATURES_V2)

print("\nPASO 59.2 COMPLETADO")
print("Datos V2 preparados para entrenamiento.")
print("No se entrenaron modelos.")
print("No se modificaron tablas Delta.")
print("No se registraron nuevos runs MLflow.")

In [0]:
# ==========================================================
# TSI - EXPERIMENTACIÓN V2
# BLOQUE 1/3: ENTRENAMIENTO Y EVALUACIÓN
# ==========================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=== EXPERIMENTACIÓN TSI V2 ===")

# 1. Función de evaluación
def evaluar(y_real, y_pred):
    return {
        "MAE": mean_absolute_error(y_real, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_real, y_pred)),
        "R2": r2_score(y_real, y_pred)
    }

# 2. Entrenamiento Ridge
print("\n=== RIDGE ===")

resultados_ridge_v2 = []

for alpha in [0.01, 0.1, 1, 10, 100]:
    modelo = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha))
    ])

    modelo.fit(X_train_v2, y_train_v2)

    pred_val = modelo.predict(X_val_v2)
    metricas = evaluar(y_val_v2, pred_val)

    resultados_ridge_v2.append({
        "alpha": alpha,
        "MAE": metricas["MAE"]
    })

mejor_alpha_v2 = min(
    resultados_ridge_v2,
    key=lambda x: x["MAE"]
)["alpha"]

modelo_ridge_v2 = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=mejor_alpha_v2))
])

modelo_ridge_v2.fit(X_train_v2, y_train_v2)

print(f"Mejor alpha: {mejor_alpha_v2}")

# 3. Entrenamiento Random Forest
print("\n=== RANDOM FOREST ===")

configuraciones = [
    (100, 5),
    (100, 10),
    (200, 5),
    (200, 10)
]

resultados_rf_v2 = []
modelos_rf_v2 = {}

for n_estimators, max_depth in configuraciones:

    modelo = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    modelo.fit(X_train_v2, y_train_v2)

    pred_val = modelo.predict(X_val_v2)
    metricas = evaluar(y_val_v2, pred_val)

    clave = (n_estimators, max_depth)
    modelos_rf_v2[clave] = modelo

    resultados_rf_v2.append({
        "config": clave,
        "MAE": metricas["MAE"]
    })

mejor_rf_config_v2 = min(
    resultados_rf_v2,
    key=lambda x: x["MAE"]
)["config"]

modelo_rf_v2 = modelos_rf_v2[mejor_rf_config_v2]

print(f"Mejor configuración: {mejor_rf_config_v2}")

# 4. Evaluación final en prueba
print("\n=== EVALUACIÓN EN TEST ===")

predicciones_v2 = {
    "Persistencia": df_test_v2_pd[
        "tsi_promedio"
    ].to_numpy(),

    "Ridge": modelo_ridge_v2.predict(X_test_v2),

    "Random Forest": modelo_rf_v2.predict(X_test_v2)
}

resultados_finales_v2 = []

for nombre, pred in predicciones_v2.items():

    metricas = evaluar(y_test_v2, pred)

    resultados_finales_v2.append({
        "Modelo": nombre,
        **metricas
    })

df_comparacion_v2 = (
    pd.DataFrame(resultados_finales_v2)
    .sort_values("MAE")
    .reset_index(drop=True)
)

print("\n=== COMPARACIÓN FINAL TSI V2 ===")
print(
    df_comparacion_v2.round(4).to_string(index=False)
)

# 5. Comparación V1 vs V2
print("\n=== COMPARACIÓN V1 VS V2 ===")

mae_rf_v1 = 1.5274

mae_rf_v2 = df_comparacion_v2.loc[
    df_comparacion_v2["Modelo"] == "Random Forest",
    "MAE"
].iloc[0]

print(f"Random Forest V1 MAE: {mae_rf_v1:.4f}")
print(f"Random Forest V2 MAE: {mae_rf_v2:.4f}")

print("\nBLOQUE 1 COMPLETADO")
print("Modelos V2 entrenados y evaluados.")
print("Sin modificaciones a Delta Lake.")
print("Sin nuevos registros MLflow.")

In [0]:
# ==========================================================
# TSI - EXPERIMENTACIÓN V2
# BLOQUE 2/3: REGISTRO EN MLFLOW
# ==========================================================

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from mlflow.models import infer_signature

EXPERIMENTO = "/Shared/TSI_PEMS_BAY_Pronostico_Horario"
mlflow.set_experiment(EXPERIMENTO)

# Usar tipos numéricos consistentes para inferencia
X_train_mlflow = X_train_v2.astype("float64")
X_example = X_train_mlflow.head(5)

# Los modelos ya están entrenados: no reentrenar
modelos = {
    "Ridge": modelo_ridge_v2,
    "Random Forest": modelo_rf_v2
}

metricas_por_modelo = (
    df_comparacion_v2
    .set_index("Modelo")
    .to_dict("index")
)

runs_v2 = {}

print("=== REGISTRO TSI V2 EN MLFLOW ===")

for nombre in ["Persistencia", "Ridge", "Random Forest"]:

    metricas = metricas_por_modelo[nombre]

    run_name = (
        "tsi_v2_"
        + nombre.lower().replace(" ", "_")
    )

    with mlflow.start_run(run_name=run_name) as run:

        mlflow.log_params({
            "proyecto": "Traffic Stability Index",
            "fuente": "pems_bay",
            "modelo": nombre,
            "version_tsi": "V2",
            "horizonte_horas": 1,
            "normalizacion": "percentiles_train_interno",
            "p05": float(p05_interno),
            "p95": float(p95_interno),
            "criterio_seleccion": "mae_validacion",
            "train_registros": len(X_train_v2),
            "validacion_registros": len(X_val_v2),
            "test_registros": len(X_test_v2)
        })

        if nombre == "Ridge":
            mlflow.log_param(
                "alpha",
                float(mejor_alpha_v2)
            )

        if nombre == "Random Forest":
            mlflow.log_params({
                "n_estimators": mejor_rf_config_v2[0],
                "max_depth": mejor_rf_config_v2[1],
                "min_samples_leaf": 5
            })

        mlflow.log_metrics({
            "mae_test": float(metricas["MAE"]),
            "rmse_test": float(metricas["RMSE"]),
            "r2_test": float(metricas["R2"])
        })

        if nombre in modelos:

            modelo = modelos[nombre]

            pred_example = modelo.predict(X_example)

            signature = infer_signature(
                X_example,
                pred_example
            )

            mlflow.sklearn.log_model(
                sk_model=modelo,
                name="modelo",
                signature=signature,
                input_example=X_example
            )

        runs_v2[nombre] = run.info.run_id

    print(f"{nombre}: {runs_v2[nombre]}")

print("\n=== REGISTRO COMPLETADO ===")
print(f"Experimento: {EXPERIMENTO}")
print("Tres runs V2 registrados.")
print("Modelos V1 conservados.")
print("Sin modificaciones a Delta Lake.")

In [0]:
# ==========================================================
# TSI - EXPERIMENTACIÓN V2
# BLOQUE 3/3: VERIFICACIÓN Y CIERRE
# ==========================================================

import mlflow
import pandas as pd
import numpy as np

print("=== VERIFICACIÓN FINAL MLFLOW V2 ===")

runs_esperados = {
    "Persistencia": "1b40d6a3cf6f4e85ad0917e39a17ea57",
    "Ridge": "936622ede13d4f8da511560515226e7f",
    "Random Forest": "3c49d369bc114c10812d0cafcdbc79da"
}

resultados = []

for nombre, run_id in runs_esperados.items():

    run = mlflow.get_run(run_id)

    assert run.info.status == "FINISHED", (
        f"Run incompleto: {nombre}"
    )

    assert run.data.params.get("version_tsi") == "V2"

    metricas = run.data.metrics

    for metrica in ["mae_test", "rmse_test", "r2_test"]:
        assert metrica in metricas
        assert np.isfinite(metricas[metrica])

    resultados.append({
        "Modelo": nombre,
        "MAE": metricas["mae_test"],
        "RMSE": metricas["rmse_test"],
        "R2": metricas["r2_test"],
        "Run ID": run_id
    })

df_final_mlflow = (
    pd.DataFrame(resultados)
    .sort_values("MAE")
    .reset_index(drop=True)
)

print("\n=== RESULTADOS VERIFICADOS ===")
print(df_final_mlflow.round(4).to_string(index=False))

mejor_modelo = df_final_mlflow.iloc[0]

assert mejor_modelo["Modelo"] == "Random Forest"

mae_baseline = df_final_mlflow.loc[
    df_final_mlflow["Modelo"] == "Persistencia",
    "MAE"
].iloc[0]

mejora = (
    (mae_baseline - mejor_modelo["MAE"])
    / mae_baseline
) * 100

print("\n=== MODELO SELECCIONADO ===")
print(f"Modelo: {mejor_modelo['Modelo']}")
print(f"MAE: {mejor_modelo['MAE']:.4f}")
print(f"RMSE: {mejor_modelo['RMSE']:.4f}")
print(f"R2: {mejor_modelo['R2']:.4f}")
print(f"Mejora frente a persistencia: {mejora:.2f}%")
print(f"Run ID: {mejor_modelo['Run ID']}")

print("\n=== ESTADO DEL PROYECTO ===")
print("Experimentación TSI V2: COMPLETADA")
print("Tres modelos comparados.")
print("Resultados verificados en MLflow.")
print("Random Forest seleccionado para siguiente fase.")
print("Tablas Delta originales conservadas.")
print("Validación local en Guadalajara: PENDIENTE.")
print("Hipótesis de anticipación 18-20 min: NO VALIDADA.")

print("\nFASE DE EXPERIMENTACIÓN CERRADA")